# float32 is OPT-IN

The switch below is `False`: this notebook runs in float64, the default. float32 is **off by default and is not a safe replacement for float64**.

- **Set it before import.** The switch below sets PyTensor's `floatX` flag, and PyTensor reads it when it is imported. If you change `USE_FLOAT32`, **restart the kernel** and run the notebook from the top. Setting it after `import pymc` does nothing.
- **Where it helps.** On a GPU, models with a lot of arithmetic per step, such as logistic regression and the non-centered radon model, run 3x to 8x faster (see `float32.ipynb`). On a CPU expect little.
- **Where it fails.** The GEV model: the step size collapses and the chains barely move. 8 Schools failed the bias gate on 2 of 13 seeds. The failure can be silent.
- **Three checks on every float32 result** (README, section "float32 on the GPU: opt-in, and check the result"):
  1. `idata.attrs["frozen_step_size"]`: orders of magnitude below the float64 value is the GEV failure;
  2. R-hat on every parameter: far above 1.01 is a failure. (With only 10 draws per chain R-hat is inflated even in float64, so compare with a float64 run of the same model; a collapsed sampler is orders of magnitude worse, not tenths.)
  3. `sample_stats["num_integration_steps"]`: hundreds of steps instead of tens.

Do **not** enable `jax_enable_x64` yourself; PyTensor manages it from `floatX`.

In [ ]:
USE_FLOAT32 = False      # the switch: read the cell above, then restart the kernel to change it

import os
if USE_FLOAT32:
    os.environ["PYTENSOR_FLAGS"] = "floatX=float32,blas__ldflags=-llapack -lblas -lcblas -lm"
print("float32 requested:", USE_FLOAT32)

# New data every call
The workflow: **one model structure, called many times** with new observed data and new prior values (a refit per day, per customer, per experiment).

The sampler compiles its programs the first time it sees a model structure, a set of array shapes, a chain count and sampler settings. Observed data and prior values are *arguments* of the compiled programs, not constants baked into them. So:

- put the observed data in `pm.Data` (the numbers can then change without changing the model graph);
- write prior values as ordinary Python numbers: they are lifted into arguments automatically;
- **rebuilding the model object for each dataset is fine**: a rebuilt model is recognised as the same structure. No need to keep one model and call `pm.set_data`.

Below, a small regression is fitted to four datasets, each with its own data and its own prior. Call 1 compiles; calls 2 to 4 compile nothing.

In [ ]:
%matplotlib inline
CHAINS, DRAWS, WARMUP = 1024, 10, 50      # many short chains; WARMUP is chees_kwargs["draws"]
SEED = 42

import jax
import numpy as np
import pymc as pm
import arviz as az
import pytensor
from pfcheeshmc import sample_pf_chees_hmc

print("JAX devices:", jax.devices(), "| floatX:", pytensor.config.floatX, "| x64:", jax.config.jax_enable_x64)
import time

In [ ]:
N = 200

def make_model(x, y, slope_prior_mean, slope_prior_sd):
    """The same structure every time: only the numbers in x, y and the prior values change."""
    with pm.Model() as model:
        x_ = pm.Data("x", x)
        y_ = pm.Data("y", y)
        a = pm.Normal("a", 0.0, 5.0)
        b = pm.Normal("b", slope_prior_mean, slope_prior_sd)
        s = pm.HalfNormal("s", 2.0)
        pm.Normal("obs", a + b * x_, s, observed=y_)
    return model

# Counts XLA programs built (or loaded from the disk cache) while a call runs.
from jax._src import dispatch
n_programs = [0]
jax.monitoring.register_event_duration_secs_listener(
    lambda event, duration_secs, **kw: n_programs.__setitem__(0, n_programs[0] + (event == dispatch.BACKEND_COMPILE_EVENT)))

In [ ]:
rng = np.random.default_rng(0)
true_slopes = [-2.0, 0.5, 1.5, 3.0]
prior_means = [0.3, 1.2, -0.7, 2.4]        # a different prior location and scale for every call
prior_sds = [3.1, 2.4, 4.2, 3.7]
rows, posteriors = [], []
for k, (slope, pm_k, ps_k) in enumerate(zip(true_slopes, prior_means, prior_sds), start=1):
    x = rng.normal(size=N)
    y = 1.0 + slope * x + rng.normal(scale=0.5, size=N)
    model = make_model(x, y, slope_prior_mean=pm_k, slope_prior_sd=ps_k)   # rebuilt every time
    n0, t0 = n_programs[0], time.perf_counter()
    with model:
        idata = sample_pf_chees_hmc(draws=DRAWS, chains=CHAINS, random_seed=k, progressbar=False,
                                    chees_kwargs={"draws": WARMUP})
    wall = time.perf_counter() - t0
    post = idata.posterior["b"]
    rows.append(dict(call=k, true_slope=slope, prior_mean=pm_k, prior_sd=ps_k, wall_s=round(wall, 2),
                     xla_programs=n_programs[0] - n0,
                     post_mean_b=round(float(post.mean()), 3), post_sd_b=round(float(post.std()), 3)))
    posteriors.append(post.values.ravel())
import pandas as pd
pd.DataFrame(rows).set_index("call")

Reading the table: `wall_s` is the wall time of the whole call, and `xla_programs` the number of XLA programs built or loaded during it. Call 1 builds them all; calls 2 to 4 build **none** and are much faster, although the data and the prior changed every time. (An exact least-squares fit gives the slope to about 0.04, so `post_mean_b` should sit within a few `post_sd_b` of `true_slope`.)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 3))
for r, draws_b in zip(rows, posteriors):
    ax.hist(draws_b, bins=40, alpha=0.6, label=f"call {r['call']}")
    ax.axvline(r["true_slope"], color="k", lw=0.8)
ax.set_xlabel("slope b (black lines: true values)"); ax.legend();

## The disk cache
The compiled programs are also written to disk by default (`${XDG_CACHE_HOME:-~/.cache}/pfcheeshmc/jax`), so the first call of a *new Python process* reads them instead of compiling: run this notebook a second time, after a kernel restart, and call 1 is much faster than the first time. The cached programs contain no data and no prior values, so a new dataset at the same shapes reuses the entries of an old one. A new shape (more observations), a new chain count or a new model structure compiles once. Details: README, "Compile once, and the disk cache". Turn it off with `PFCHEESHMC_COMPILATION_CACHE=0`.